In [4]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
_='''
read data from semi_processed

select is_good_peak == 3

see interval_id

devid interval to n batch (each k interval) each batch [i1,i2)

write class model that get intreval of interval id [i1,i2) and run model over interval [i1,i2)
model get interval [i3,i4) and test model over it

for each batch train Model
and test it over interval after and before

if error < threshould then merge to batch and train new model and compute error for before and after batch

do this work until no error least than thresould
'''

In [26]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from src.models.utils import *
# Placeholder for your actual model class
class Model:
    def __init__(self, interval, data):
        self.i_start = interval[0]
        self.i_end = interval[1]
        self.data = get_in(data,(self.i_start,self.i_end))  # data filtered for this interval
        self.model = None
        self.n_mimo = 1

    def train(self):
        # TODO: Implement model training on self.data
        
        train_data = self.data
        train_data = train_data.drop(columns=['datetime','name','code'])
        X = train_data.drop(columns=["generation"])
        y = train_data["generation"]
        
        model = LinearRegression()
        model.fit(X,y)
        self.model = model
        
        #print(f"Training model on interval [{self.i_start}, {self.i_end}) with {len(self.data)} rows")
        

    def test(self, interval):
        test_data = get_in(self.data,interval)
        test_data = test_data.drop(columns=['datetime','name','code'])
        X = test_data.drop(columns=["generation"])
        y = test_data["generation"]
    
        y_pred = self.model.predict(X)
        rmse_error_test = compute_relative_rmse(y_pred, y)
        #print(f"Testing model trained on [{self.i_start}, {self.i_end}) on test data with {len(test_data)} rows")
        return rmse_error_test


def split_intervals(intervals, batch_size):
    """Split sorted unique interval IDs into batches of size batch_size"""
    batches = []
    for i in range(0, len(intervals), batch_size):
        batches.append((i,i+batch_size))
    batches[-1] = (batches[-1][0],len(intervals))
    return batches

def get_in(df,i):
    (i1,i2) = i
    mask = (i1 <= df['interval_id']) & (df['interval_id'] <= i2)
    return df[mask]

In [27]:
pd.set_option('future.no_silent_downcasting', True)
# Load your data
# TODO: Replace with actual data reading logic
# df = pd.read_csv('semi_processed.csv') or pd.read_parquet('semi_processed.parquet')
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')

name = "پرند"
code = "G11"
# Select rows where is_good_peak == 3
filtered_df = df[(df["name"] == name)&(df["code"] == code)&(df['is_good_peak'] == 3)]
filtered_df = filtered_df[["name","code","datetime","generation",'interval_id',"temperature", "humidity", "dew", "surface_pressure"]]
# Unique sorted interval IDs
unique_intervals = sorted(filtered_df['interval_id'].unique())

batch_size = 20  # For example, number of intervals per batch
batches = split_intervals(unique_intervals, batch_size)

threshold = 0.1  # Define your error threshold

batch_models = []
errors = [[None,None] for i in range(len(batches))]

for idx, batch_intervals in enumerate(batches):
    i_start = batch_intervals[0]
    i_end = batch_intervals[1]  # Exclusive range assumption

    # Train model on current batch
    batch_model = Model(batch_intervals, filtered_df)
    batch_model.train()

    # Test model on before and after intervals
    error_before = None
    error_after = None

    if idx > 0:
        # Test on previous batch interval
        prev_intervals = batches[idx-1]
        error_before = batch_model.test(prev_intervals)
        errors[idx][0] = error_before

    if idx < len(batches) - 1:
        # Test on next batch interval
        next_intervals = batches[idx+1]
        error_after = batch_model.test(next_intervals)
        errors[idx][1] = error_after

    #print(f"Batch {idx}: error_before={error_before}, error_after={error_after}")

    batch_models.append(batch_model)

# Repeat merging & retraining logic until no error less than threshold
# You can wrap above in a loop and add conditions accordingly

In [35]:
def merge_batches(batches, batch_models, df, errors, threshold):
    """
    Merge adjacent batches in the 'batches' list when the error between their models is below 'threshold'.
    
    Parameters:
    batches (list of list): Each inner list contains interval_ids representing one batch.
    batch_models (list): List of trained Model instances corresponding to batches.
    df (pd.DataFrame): The dataframe containing the data with 'interval_id' column.
    threshold (float): Error threshold to decide whether to merge batches.
    
    Returns:
    batches (list of list): Updated list of batches after merging.
    batch_models (list): Updated list of models after retraining on merged batches.
    merged (bool): True if any merge was performed, otherwise False.
    """
    
    merged = False
    i = 0
    # Step 2: Iterate over computed errors and merge batches with error below threshold
    while i < len(errors):
        if errors[i][1] is not None and errors[i][1] < threshold:
            # Merge batches i and i+1
            new_batch = (batches[i][0],batches[i+1][1])

            # Train a new model on the merged data
            new_model = Model(new_batch, df)
            new_model.train()
            print("Train a new model on the merged data")
            # Update batches and models lists by replacing merged batches with the new one
            batches[i] = new_batch
            batch_models[i] = new_model
            del batches[i+1]
            del batch_models[i+1]
            
            e1 = batch_models[i].test(batches[i+1]) if i < len(errors)-1 else None
            e2 = batch_models[i].test(batches[i-1]) if i > 0 else None
            errors[i] = [e1, e2]
            
            if i < len(errors)-1:
                errors.pop(i+1)
                e = batch_models[i+1].test(batches[i])
                errors[i+1][0] = e
                
            if i > 0:
                e = batch_models[i-1].test(batches[i])
                errors[i-1][1] = e
            
            '''
            errors[i] = test new model on pre and next batch
            del errors[i+1]
            errors[i-1][1] = test model[i-1] on new batch
            errors[i+1][0] = test model[i+1] on new batch
            '''
            merged = True
            break
        i += 1

    return merged,i


# -------- Example of how to run the merge process --------

threshold = 1 # Define your error threshold
is_merged = True

print(len(batches))
is_merged,i = merge_batches(batches, batch_models, filtered_df,errors, threshold)
print(len(batches),i)
print("Merging process completed.",is_merged)


21
Train a new model on the merged data
20 10
Merging process completed. True
